# Week 2: Data Audit and Study Population

## Project Title
Predicting Non-Use of Skilled Birth Attendance in Nigeria Using Pre-Delivery Information: Evidence from the 2024 Nigeria Demographic and Health Survey

## Objectives
1. Inspect the structure and quality of the NDHS Births Recode dataset.
2. Define the eligible study population.
3. Examine the skilled birth attendance outcome.
4. Audit candidate predictors and missing values.
5. Document data-quality findings and methodological decisions.

## Data Source
2023–24 Nigeria Demographic and Health Survey (NDHS)

Primary dataset: Births Recode (BR)

## Study Population
Women whose most recent live birth occurred within 24 months before the survey interview.

Provisional sample size: 10,522 women.

## Data Confidentiality
The raw NDHS respondent-level data are stored locally and will not be uploaded to GitHub.



## 1. Import Libraries and Load Data

In this section, we import pandas and load the 2023–24 Nigeria DHS Births Recode dataset.

The raw dataset is stored locally and will not be uploaded to GitHub.

In [ ]:
import pandas as pd
from pathlib import Path

# Locate the project folder and raw dataset
project_root = Path.cwd().parent
data_path = project_root / "data" / "raw" / "NGBR8BFL.dta"

# Load the Births Recode dataset
df = pd.read_stata(data_path, convert_categoricals=False)

# Check the dataset dimensions
print("Total records:", df.shape[0])
print("Total columns:", df.shape[1])

## 2. Define the Study Population

The study includes women whose most recent live birth occurred within 24 months before the survey interview.

Selection criteria:
- Child's age (`b19`) is between 0 and 23 months.
- Birth index (`bidx`) equals 1, identifying the woman's most recent birth record.

This produces one index birth record per woman. For multiple births, only one child record is selected to represent the delivery.

In [ ]:
# Select births within the last 24 months
recent_births = df[df["b19"].between(0, 23)].copy()

# Select the most recent birth record per woman
study_df = recent_births[recent_births["bidx"] == 1].copy()

# Display study population counts
print("Total birth records:", len(df))
print("Births within 24 months:", len(recent_births))
print("Selected study records:", len(study_df))
print("Unique women:", study_df["caseid"].nunique())

## 3. Define the Outcome Variable

The outcome is non-use of skilled birth attendance during the most recent delivery.

For this analysis, a skilled attendant is defined as a doctor (`m3a`) or nurse/midwife (`m3b`).

Outcome coding:
- 1 = No skilled birth attendant
- 0 = Skilled birth attendant present

This definition is provisional pending final verification against the 2023–24 NDHS report.

In [ ]:
# Identify deliveries assisted by a doctor or nurse/midwife
skilled_attendant = (
    (study_df["m3a"] == 1) |
    (study_df["m3b"] == 1)
)

# Create the target variable: 1 = non-use, 0 = skilled attendance
study_df["non_use_sba"] = (~skilled_attendant).astype(int)

# Check the outcome distribution
print("Skilled attendance:", (study_df["non_use_sba"] == 0).sum())
print("Non-use of skilled attendance:", (study_df["non_use_sba"] == 1).sum())
print("Total:", len(study_df))
print("Missing outcome:", study_df["non_use_sba"].isna().sum())

## 4. Distribution of Skilled Birth Attendance

This section calculates the unweighted and survey-weighted percentages of skilled birth attendance and non-use.

The women's individual sampling weight (`v005`) is divided by 1,000,000 before use.

Weighted percentages account for unequal sampling weights. Confidence intervals and statistical inference will require additional consideration of the survey clusters and strata.

In [ ]:
# Convert DHS sampling weights
study_df["sample_weight"] = study_df["v005"] / 1_000_000

# Calculate unweighted percentages
unweighted = (
    study_df["non_use_sba"]
    .value_counts(normalize=True)
    .sort_index() * 100
)

# Calculate survey-weighted percentages
weighted = (
    study_df.groupby("non_use_sba")["sample_weight"].sum()
    / study_df["sample_weight"].sum() * 100
)

# Display results
for value, label in [(0, "Skilled attendance"), (1, "Non-use")]:
    print(
        f"{label}: "
        f"Unweighted = {unweighted[value]:.2f}%, "
        f"Weighted = {weighted[value]:.2f}%"
    )

## 5. Survey Sampling Design

The 2023–24 Nigeria DHS uses a complex survey sampling design.

The following variables are retained:

- `v005`: Women's individual sampling weight
- `v021`: Primary sampling unit (cluster)
- `v022`: Sampling stratum

The sampling weight is divided by 1,000,000.

Survey weights are used for weighted descriptive estimates. Appropriate survey-design methods will be required for confidence intervals and statistical inference.

In [ ]:
# Check the sampling design variables
design_vars = ["v005", "v021", "v022"]

print("Selected women:", len(study_df))
print("Unique clusters:", study_df["v021"].nunique())
print("Unique strata:", study_df["v022"].nunique())

# Check missing values
print("\nMissing values:")
print(study_df[design_vars].isna().sum())

# Check sampling weights
print("\nZero weights:", (study_df["v005"] == 0).sum())

## 6. Candidate Predictor Audit

This section examines the completeness of potential predictors of non-use of skilled birth attendance.

Candidate variables include:

- Maternal age at delivery
- Geopolitical zone
- Urban/rural residence
- Maternal education
- Household wealth index
- Marital status
- Number of antenatal care visits
- Timing of first antenatal care visit

These are provisional predictors. Their availability before delivery and potential for data leakage will be assessed before final model selection.

Special DHS response codes, such as 98 (don't know), must not be interpreted as ordinary numeric values.

In [ ]:
# Candidate predictor variables
predictors = {
    "v012": "Age at interview",
    "v024": "Geopolitical zone",
    "v025": "Residence",
    "v106": "Education",
    "v190": "Wealth index",
    "v501": "Marital status",
    "m14": "Number of ANC visits",
    "m13": "Timing of first ANC visit"
}

# Create a missing-value summary
audit = pd.DataFrame({
    "Variable": list(predictors.keys()),
    "Description": list(predictors.values()),
    "Missing": [
        study_df[col].isna().sum()
        for col in predictors
    ]
})

audit["Missing (%)"] = (
    audit["Missing"] / len(study_df) * 100
).round(2)

display(audit)

## 7. Special Codes and Structural Missingness

DHS variables may contain special response codes that pandas does not automatically recognize as missing.

For ANC variables:
- `m14 = 98`: Unknown number of ANC visits.
- `m13 = 98`: Unknown timing of first ANC visit.
- `m13 = NaN` with `m14 = 0`: Structural missingness because no ANC visit occurred.

These categories will be preserved during the audit. No respondents will be excluded solely because of these values.

In [ ]:
# Identify ANC special codes and structural missingness

print("Unknown ANC visit count (m14 = 98):",
      (study_df["m14"] == 98).sum())

print("Unknown first ANC timing (m13 = 98):",
      (study_df["m13"] == 98).sum())

print("Structural missingness in m13:",
      (
          study_df["m13"].isna()
          & (study_df["m14"] == 0)
      ).sum())

print("Other missing m13 values:",
      (
          study_df["m13"].isna()
          & (study_df["m14"] != 0)
      ).sum())

## 8. Maternal Age at Delivery

Maternal age at delivery is calculated using:

- `v011`: Woman's date of birth (CMC)
- `b3`: Child's date of birth (CMC)

Age at delivery = (b3 - v011) / 12

This variable is preferred over age at interview (`v012`) because our prediction study focuses on information available by the time of delivery.

Since CMC dates are recorded in months, the calculated age is approximate.

Initial plausibility checks will identify missing values and unusually low or high ages.

In [ ]:
# Calculate approximate maternal age at delivery
study_df["age_at_delivery"] = (
    study_df["b3"] - study_df["v011"]
) / 12

# Summarize maternal age at delivery
print("Selected women:", len(study_df))
print("Minimum age:", round(study_df["age_at_delivery"].min(), 2))
print("Maximum age:", round(study_df["age_at_delivery"].max(), 2))
print("Average age:", round(study_df["age_at_delivery"].mean(), 2))
print("Missing ages:", study_df["age_at_delivery"].isna().sum())

# Check unusual values
print("Age below 15:", (study_df["age_at_delivery"] < 15).sum())
print("Age above 49:", (study_df["age_at_delivery"] > 49).sum())

## 9. Predictor Eligibility and Data Leakage Assessment

The study aims to predict non-use of skilled birth attendance using information available before delivery.

A predictor must satisfy two conditions:

1. It must be available at the intended prediction time.
2. It must not contain information obtained only after delivery.

### Preliminary Predictor Assessment

| Variable | Description | Preliminary Decision |
|---|---|---|
| age_at_delivery | Maternal age at delivery | Retain provisionally; prediction timing must be defined |
| v024 | Geopolitical zone | Review residence timing |
| v025 | Urban/rural residence | Review residence timing |
| v106 | Maternal education | Review whether education changed after delivery |
| v190 | Household wealth index | Review interview-versus-delivery timing |
| v501 | Marital status at interview | Do not include without further justification |
| m14 | Total ANC visits | Conditional on prediction timing |
| m13 | Timing of first ANC visit | Conditional on prediction timing |

### Research Decision

No final predictor set has been selected.

Variables measured at interview may not accurately represent circumstances before delivery.

ANC variables may be suitable for a late-pregnancy prediction model but not necessarily for an early-pregnancy model.

The intended prediction time must be established before final feature selection.

## 10. Prediction Time and Intended Use

### Prediction Time

The intended prediction period is the third trimester of pregnancy, approximately 28–36 weeks of gestation.

### Research Objective

To identify pregnant women at increased risk of delivering without a skilled birth attendant, allowing healthcare providers and public health programmes to consider targeted support before childbirth.

### Predictor Eligibility

Only information that would reasonably be available by the prediction time should be used.

Potential predictors include:

- Maternal age at the prediction time
- Geographical location
- Urban or rural residence
- Maternal education
- Household socioeconomic circumstances
- Antenatal care information available by the prediction time

### Data Leakage Prevention

The following precautions will be applied:

1. Information obtained only after delivery will not be used as a predictor.
2. Total ANC visits across the entire pregnancy will not automatically be included.
3. Timing of first ANC attendance will be considered only if the visit occurred before the prediction time.
4. Variables recorded at the survey interview will be assessed for possible differences from their values during pregnancy.
5. The final predictor set will be justified and documented before modelling.

### Study Limitation

The NDHS data were collected retrospectively. Some variables describe circumstances at the survey interview rather than at the intended prediction time.

The study will therefore be treated as a retrospective prediction analysis, not a prospective clinical validation.

## 11. Candidate Predictor Data Dictionary

This section documents the candidate predictors identified during the Week 2 data audit.

The dictionary records:
- Variable name
- Description
- Data type
- Preliminary eligibility for third-trimester prediction

These variables are not yet the final modelling features.

In [ ]:
# Create a preliminary predictor data dictionary

data_dictionary = pd.DataFrame([
    ["age_at_delivery", "Approximate maternal age at delivery",
     "Numeric", "Review: use age at prediction time"],

    ["v024", "Geopolitical zone",
     "Categorical", "Review residence timing"],

    ["v025", "Urban/rural residence",
     "Categorical", "Review residence timing"],

    ["v106", "Highest educational level",
     "Categorical", "Review education timing"],

    ["v190", "Household wealth quintile",
     "Ordinal", "Review interview timing"],

    ["v501", "Marital status at interview",
     "Categorical", "Exclude provisionally"],

    ["m14", "Total ANC visits",
     "Numeric", "Exclude total visits provisionally"],

    ["m13", "Month of first ANC visit",
     "Numeric", "Conditional on prediction timing"]
], columns=[
    "Variable",
    "Description",
    "Data Type",
    "Preliminary Decision"
])

display(data_dictionary)

## 12. Week 2 Data Audit Summary

### Dataset Overview
- Data source: 2023–24 Nigeria Demographic and Health Survey (NDHS).
- Primary dataset: Births Recode (BR).
- Original dataset: 104,557 birth records and 1,336 variables.
- Birth records within the previous 24 months: 10,970.
- Selected study population: 10,522 women.
- Selection method: Most recent live birth (`bidx = 1`).

### Outcome Variable
- Target: Non-use of skilled birth attendance.
- Skilled attendance: 5,028 women.
- Non-use: 5,494 women.
- Weighted non-use estimate: 54.91%.
- Outcome definition remains subject to final documentation verification.

### Sampling Design
- Primary sampling units: 1,364.
- Sampling strata: 74.
- Missing sampling-design values: 0.
- Sampling weight: `v005 / 1,000,000`.

### Maternal Age
- Average age at delivery: 28.38 years.
- Minimum age: 14.50 years.
- Maximum age: 49.17 years.
- Missing values: 0.
- Three unusual ages were reviewed and provisionally retained.

### Antenatal Care
- No ANC visits: 2,777 women.
- Reported 1–20 visits: 7,459 women.
- Unknown visit count: 286 women.
- Missing first-ANC timing among women with no ANC: 2,777.
- Unaccounted records in the ANC consistency audit: 0.

### Predictor Eligibility
The intended prediction period is the third trimester.

Variables measured after the prediction time must not be used without justification.

Current marital status and total ANC visits are excluded provisionally from the modelling feature set.

### Data Quality Decision
All 10,522 selected women are retained at this stage.

No records have been removed during the exploratory data audit.

### Outstanding Decisions
1. Verify the outcome definition against official NDHS documentation.
2. Finalize the exact third-trimester prediction point.
3. Confirm which predictors were available at that point.
4. Establish the modelling and validation strategy.
5. Document limitations arising from retrospective survey measurements.